# LangChain Expression Language (LCEL) with Ollama
This notebook provides hands-on, executable examples of the LCEL concepts covered in the [AI Engineering Visualized guide](https://amara-manikanta.github.io/ai-engineering-visualized/agents/langchain.html).

Ensure you have Ollama installed locally and have pulled the `llama3` model before running:
`ollama run llama3`

In [ ]:
!pip install langchain-core langchain-ollama

## 1. Basic LCEL Chain (Pipe Operator)

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama.llms import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate.from_template("Tell me a short, 1-sentence joke about {topic}")
model = OllamaLLM(model="llama3")
parser = StrOutputParser()

# Compose the chain using the pipe operator
chain = prompt | model | parser

# Invoke the chain
response = chain.invoke({"topic": "cats"})
print(response)

## 2. Streaming

In [ ]:
for chunk in chain.stream({"topic": "bears"}):
    print(chunk, end="", flush=True)

## 3. Batching

In [ ]:
responses = chain.batch([
    {"topic": "bears"}, 
    {"topic": "cats"}, 
    {"topic": "dogs"}
])

for i, resp in enumerate(responses):
    print(f"Joke {i+1}: {resp}")

## 4. Runnable Parallel

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

def fake_retriever(query: str):
    return "Bears love eating honey and catching salmon."

parallel_setup = RunnableParallel(
    context=fake_retriever, 
    topic=RunnablePassthrough()
)

rag_prompt = PromptTemplate.from_template("Context: {context}\n\nTell me a joke about: {topic}")

rag_chain = parallel_setup | rag_prompt | model | parser

print(rag_chain.invoke("bears"))